# 🎨 VecGlypher — Генерация векторных глифов
Запускайте ячейки по порядку сверху вниз. Каждую ячейку запускать кнопкой ▶ или `Shift+Enter`.

---
## Шаг 1 — Настройки
Укажите путь к модели и порт сервера. Запустите один раз в начале.

In [ ]:
import requests, json, os
from IPython.display import display, HTML

# ============================================================
# НАСТРОЙКИ — измените под себя
# ============================================================
MODEL_PATH = "/workspace/VecGlypher/saves/VecGlypher-27b-it"
SERVER_PORT = 30000
SERVER_URL  = f"http://localhost:{SERVER_PORT}/v1"
OUTPUT_DIR  = "/workspace/VecGlypher/outputs"

# Параметры генерации
TEMPERATURE        = 0.7
TOP_P              = 0.8
TOP_K              = 20
MAX_TOKENS         = 1024
REPETITION_PENALTY = 1.05

SYSTEM_PROMPT = """You are a specialized vector glyph designer creating SVG path elements.

CRITICAL REQUIREMENTS:
- Each glyph must be a complete, self-contained element, in reading order of the given text.
- Terminate each element with a newline character
- Output ONLY valid SVG elements"""

os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_glyph(character, font_style):
    """Генерирует один глиф и сразу сохраняет SVG файл."""
    user_prompt = f"Font design requirements: {font_style}\nText content: {character}"
    response = requests.post(
        f"{SERVER_URL}/chat/completions",
        headers={"Content-Type": "application/json", "Authorization": "dummy-key"},
        json={
            "model": MODEL_PATH,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "top_k": TOP_K,
            "min_p": 0.0,
            "repetition_penalty": REPETITION_PENALTY,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": MAX_TOKENS
        },
        timeout=120
    )
    svg_paths = response.json()["choices"][0]["message"]["content"]
    full_svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000" width="300" height="300">\n  <rect width="1000" height="1000" fill="white"/>\n  <g fill="black" stroke="none">{svg_paths}</g>\n</svg>'

    # Сохраняем сразу после генерации
    safe_style = font_style[:20].replace(' ', '_').replace(',', '')
    filename = f"{OUTPUT_DIR}/glyph_{character}_{safe_style}.svg"
    with open(filename, "w") as f:
        f.write(full_svg)

    return full_svg, filename

print("✅ Настройки загружены")
print(f"   Модель : {MODEL_PATH}")
print(f"   Сервер : {SERVER_URL}")
print(f"   Папка  : {OUTPUT_DIR}")

---
## Шаг 2 — Проверка сервера
Убедитесь что vLLM запущен перед генерацией.

In [ ]:
try:
    r = requests.get(f"{SERVER_URL}/models", timeout=3)
    models = r.json()
    print("✅ Сервер работает!")
    print(f"   Модель: {models['data'][0]['id']}")
except Exception as e:
    print(f"❌ Сервер недоступен: {e}")
    print("   Запустите vLLM: vllm serve /workspace/VecGlypher/saves/VecGlypher-27b-it --port 30000")

---
## Шаг 3 — Генерация одного символа
Глиф сохраняется сразу после генерации.

In [ ]:
# ============================================================
# ✏️ МЕНЯЙТЕ ЭТИ ПАРАМЕТРЫ
# ============================================================
CHARACTER  = "A"
FONT_STYLE = "humanist sans-serif, 600 weight, calm, competent, business"
# ============================================================

print(f"🎨 Генерируем '{CHARACTER}'...")
full_svg, filename = generate_glyph(CHARACTER, FONT_STYLE)
print(f"✅ Сохранено: {filename}")
print("🖼️ Результат:")
display(HTML(full_svg))

---
## Шаг 4 — Генерация нескольких символов
Каждый символ сохраняется **сразу** после генерации — даже если процесс прервётся, уже готовые файлы останутся.

In [ ]:
# ============================================================
# ✏️ МЕНЯЙТЕ ЭТИ ПАРАМЕТРЫ
# ============================================================
CHARACTERS_BATCH = ["A", "B", "C", "D", "E"]
FONT_STYLE_BATCH = "elegant serif, thin weight, luxury, fashion"
# ============================================================

results_html = "<div style='display:flex; flex-wrap:wrap; gap:15px; padding:10px;'>"
saved = []

for char in CHARACTERS_BATCH:
    print(f"  🎨 Генерируем '{char}'...", end=" ", flush=True)
    try:
        full_svg, filename = generate_glyph(char, FONT_STYLE_BATCH)
        saved.append(filename)
        # Уменьшаем для отображения
        svg_small = full_svg.replace('width="300" height="300"', 'width="150" height="150"')
        results_html += f"<div style='text-align:center'>{svg_small}<br><b>{char}</b></div>"
        print(f"✅ сохранён: {filename}")
    except Exception as e:
        print(f"❌ ошибка: {e}")

results_html += "</div>"
print(f"\n🏁 Готово! Сохранено {len(saved)} из {len(CHARACTERS_BATCH)} глифов")
print("🖼️ Результаты:")
display(HTML(results_html))

---
## Утилиты

In [ ]:
# Показать все сохранённые файлы
files = sorted(os.listdir(OUTPUT_DIR))
print(f"📁 Файлы в {OUTPUT_DIR} ({len(files)} шт.):")
for f in files:
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"   {f} ({size} байт)")

In [ ]:
# Проверить GPU
import subprocess
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.used,memory.free,utilization.gpu --format=csv,noheader"))